# Midterm 1: run the data generator in Colab


In [1]:
GITHUB_USERNAME = "aidashunayeva-sys"   # e.g. "aigerim-k"
CODE = "KETTLE-62"      # replace with the session code during the exam
assert GITHUB_USERNAME, "fill in GITHUB_USERNAME first"

In [2]:
import os

if not os.path.exists("midterm-1-AI-course"):
    !git clone --quiet https://github.com/unreal-kz/midterm-1-AI-course.git
%cd midterm-1-AI-course
!python make_my_data.py --github {GITHUB_USERNAME} --code {CODE}

/content/midterm-1-AI-course
Wrote 3 files to my_data/ · fingerprint 687a0422
Next: open my_data/DATA.md


In [3]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(open("my_data/DATA.md", encoding="utf-8").read()))
ranks = pd.read_csv("my_data/retrieval_ranks.csv")   # Part 1
docs = pd.read_csv("my_data/doc_results.csv")        # Part 2
display(ranks.head(), docs)

# Midterm 1: your data

GitHub username: `aidashunayeva-sys` · session code: `kettle-62` · generator v1

**Fingerprint: `687a0422`**: copy it into the first lines of `ANSWERS.md`. All numbers below are synthetic: this is an exam, not a measurement.

## Part 1: inputs

`retrieval_ranks.csv`: 60 test questions (20 EN, 20 RU, 20 KK). `rank_A` and `rank_B` give the position of the correct chunk in the top 10 retrieved under config A and config B; **11 means the correct chunk was not in the top 10.**

Production traffic and Model M token counts (tokens per item, as counted by Model M's tokenizer):

| language | share of traffic | question tokens | answer tokens | chunk A tokens | chunk B tokens |
|---|---|---|---|---|---|
| EN | 23 % | 37 | 101 | 264 | 73 |
| RU | 37 % | 70 | 192 | 502 | 139 |
| KK | 40 % | 80 | 217 | 568 | 157 |

- Requests per day: **6,500**, in the traffic shares above.
- System prompt: **205** tokens in every request.
- Price: **$3** per million input tokens, **$15** per million output tokens.
- Annual budget for this service (model calls only): **$21,900**.

## Part 2: inputs

`doc_results.csv`: for each pipeline, language and scan condition, `n_docs` test invoices and `n_correct` = invoices whose number, date, seller ID and total were all extracted exactly (strict scorer, as in Lab 04).

- **Pipeline H**: hosted vision-language API in a data centre abroad, pay per token; each page is sent as an image.
- **Pipeline L**: open vision-language model on your own server in Kazakhstan, fixed monthly cost.

- Volume: **3,000** scanned pages per day. In **25 %** of them an individual's name and ИИН are printed (sole proprietors, receipts).
- Page image: **1240 × 1754** pixels. Pipeline H counts an image as ⌈width ÷ 28⌉ × ⌈height ÷ 28⌉ input tokens and returns about **343** output tokens per page (the JSON).
- Pipeline H price: **$1** per million input tokens, **$5** per million output tokens.
- Pipeline L: one server costs **$900** per month and handles up to **8,000** pages per day; add servers as needed.

## Part 3: your five questions

Answer each in `ANSWERS.md`: one **Number** line and one **Why** line of at most 40 words.

### Q1

The same support request, once in English and once in Kazakh, through Model M ($3 per million input tokens, $18 per million output tokens):

| | input tokens | output tokens |
|---|---|---|
| EN | 1,576 | 138 |
| KK | 4,445 | 216 |

Compute the token ratio (total tokens KK ÷ EN) and the bill ratio (cost KK ÷ cost EN). **Number:** the bill ratio (2 decimals). **Why** (at most 40 words): why is the bill ratio not equal to the token ratio?

### Q2

A teammate sets the chunk size to **576 tokens, counted with cl100k**. The retrieval model (e5-small) reads at most 512 of *its own* tokens. Assume e5 tokens grow in proportion to cl100k tokens at the rate measured in Lecture 4.1 (a chunker told 256 cl100k tokens produced chunks of 263 / 128 / 76 e5 tokens for EN / RU / KK), and that every chunk is exactly the full size.

**Number:** the share of an English chunk, in percent (1 decimal), that the embedding never sees. **Why** (at most 40 words): which of the three languages are cut, and what happens to the cut part?

### Q3

A language model gives these next-token scores for the prompt "The branch is closed on":

| token | score |
|---|---|
| Monday | 4.3 |
| Friday | 3.5 |
| Sunday | 2.3 |
| holiday | 1.0 |
| Easter | 0.6 |

Use temperature T = 1, then apply top-p = 0.9. **Number:** the probability of the top token after top-p and renormalisation, in percent (1 decimal). **Why** (at most 40 words): how does top-p differ from lowering the temperature?

### Q4

Two models answer the same 100 factual questions (1 point per correct answer on the public leaderboard):

| | correct | wrong | "I don't know" |
|---|---|---|---|
| Model X | 74 | 26 | 0 |
| Model Y | 69 | 19 | 12 |

Now score them again with a penalty: +1 per correct answer, −2 per wrong answer, 0 for "I don't know". **Number:** Y's penalised score minus X's (negative if X is ahead). **Why** (at most 40 words): which model does the public leaderboard prefer, and what does that teach a model to do?

### Q5

A lab trains a model with **70 billion parameters on 26.18 trillion tokens**. Chinchilla (Lecture 1.1) trained 70 billion parameters on 1.4 trillion tokens, which the paper presents as the compute-optimal ratio. **Number:** this model's tokens per parameter divided by Chinchilla's tokens per parameter (2 decimals). **Why** (at most 40 words): is the model under- or over-trained by that criterion, and what did Chinchilla say about models such as GPT-3?


,qid,lang,rank_A,rank_B
0,q01,KK,2,5
1,q02,RU,3,5
2,q03,EN,1,3
3,q04,EN,1,1
4,q05,RU,1,1


,pipeline,lang,condition,n_docs,n_correct
0,H,EN,clean,30,29
1,H,EN,degraded,30,27
2,H,RU,clean,30,28
3,H,RU,degraded,30,26
4,H,KK,clean,30,28
5,H,KK,degraded,30,20
6,L,EN,clean,30,28
7,L,EN,degraded,30,25
8,L,RU,clean,30,27
9,L,RU,degraded,30,25


In [4]:
import pandas as pd
import numpy as np

ranks = pd.read_csv("my_data/retrieval_ranks.csv")
docs = pd.read_csv("my_data/doc_results.csv")

def wilson(k, n, z=1.96):
    if n == 0: return 0, 0
    p = k / n
    denom = 1 + z**2 / n
    center = p + z**2 / (2 * n)
    dev = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    return round((center - dev) / denom, 4), round((center + dev) / denom, 4)

print("=== PART 1a: Metrics ===")
for config in ['A', 'B']:
    col = f'rank_{config}'
    r1 = (ranks[col] == 1).mean()
    r3 = (ranks[col] <= 3).mean()
    r5 = (ranks[col] <= 5).mean()
    mrr = (1 / ranks[col].clip(upper=10)).where(ranks[col] <= 10, 0).mean()
    print(f"Config {config}: recall@1={r1:.4f}, recall@3={r3:.4f}, recall@5={r5:.4f}, MRR@10={mrr:.4f}")

print("\n=== PART 1b: Wilson Intervals ===")
N = len(ranks) # 60
for config, k_val in [('A', 3), ('A', 5), ('B', 3), ('B', 5)]:
    hits = (ranks[f'rank_{config}'] <= k_val).sum()
    rec = hits / N
    low, high = wilson(hits, N)
    print(f"Candidate {config}, k={k_val}: recall={rec:.4f}, lower={low}, upper={high}")

print("\n=== PART 1c & 1d: Language Recalls & Costs ===")
# Затраты на токены
# EN: 23%, RU: 37%, KK: 40%
# Requests: 6500/day = 2,372,500/year
req_year = 6500 * 365
for config, chunk_col in [('A', 'chunk A tokens'), ('B', 'chunk B tokens')]:
    for k_val in [3, 5]:
        # Входные токены
        # EN: 37, RU: 70, KK: 80; System: 205
        # Chunk_A: EN: 264, RU: 502, KK: 568
        # Chunk_B: EN: 73, RU: 139, KK: 157
        # Output: EN: 101, RU: 192, KK: 217
        inp_EN = 205 + 37 + k_val * (264 if config=='A' else 73)
        inp_RU = 205 + 70 + k_val * (502 if config=='A' else 139)
        inp_KK = 205 + 80 + k_val * (568 if config=='A' else 157)

        avg_inp = 0.23 * inp_EN + 0.37 * inp_RU + 0.40 * inp_KK
        avg_out = 0.23 * 101 + 0.37 * 192 + 0.40 * 217

        annual_cost = (req_year * avg_inp / 1e6 * 3) + (req_year * avg_out / 1e6 * 15)
        print(f"Cost {config}, k={k_val}: ${annual_cost:.2f}")

for config in ['A', 'B']:
    for lang in ['EN', 'RU', 'KK']:
        sub = ranks[ranks['lang'] == lang]
        for k_val in [3, 5]:
            r = (sub[f'rank_{config}'] <= k_val).mean()
            print(f"Recall {config} k={k_val} [{lang}]: {r:.4f}")

print("\n=== PART 2a: Doc Pipeline ===")
for pipe in ['H', 'L']:
    sub = docs[docs['pipeline'] == pipe]
    tot_docs = sub['n_docs'].sum()
    tot_corr = sub['n_correct'].sum()
    acc = tot_corr / tot_docs
    print(f"Pipeline {pipe} Overall Acc: {acc:.4f} ({tot_corr}/{tot_docs})")

docs['acc'] = docs['n_correct'] / docs['n_docs']
weakest = docs.loc[docs['acc'].idxmin()]
w_low, w_high = wilson(weakest['n_correct'], weakest['n_docs'])
print(f"Weakest cell: Pipeline {weakest['pipeline']}, {weakest['lang']}, {weakest['condition']} -> Acc: {weakest['acc']:.4f}, Wilson: [{w_low}, {w_high}]")

abroad_pages = 3000 * 365 * 0.25
print(f"Personal data pages abroad under H: {abroad_pages:.0f}")

print("\n=== PART 3: Questions ===")
# Q1
cost_EN = (1576 * 3 + 138 * 18) / 1000
cost_KK = (4445 * 3 + 216 * 18) / 1000
bill_ratio = cost_KK / cost_EN
print(f"Q1 Bill ratio: {bill_ratio:.2f}")

# Q2
# cl100k -> e5 tokens ratio: 263/256 = 1.02734375
e5_tokens = 576 * (263 / 256)
cut_tokens = max(0, e5_tokens - 512)
cut_share = (cut_tokens / e5_tokens) * 100
print(f"Q2 Cut share EN: {cut_share:.1f}%")

# Q3
# Scores: Monday: 4.3, Friday: 3.5, Sunday: 2.3, holiday: 1.0, Easter: 0.6
scores = np.array([4.3, 3.5, 2.3, 1.0, 0.6])
exp_s = np.exp(scores)
probs = exp_s / np.sum(exp_s)
# top-p = 0.9
cum_probs = np.cumsum(probs)
cutoff_idx = np.where(cum_probs >= 0.9)[0][0] + 1
renorm_p = probs[:cutoff_idx] / np.sum(probs[:cutoff_idx])
print(f"Q3 Top token prob: {renorm_p[0]*100:.1f}%")

# Q5
ratio = (26.18e12 / 70e9) / (1.4e12 / 70e9)
print(f"Q5 Ratio: {ratio:.2f}")

=== PART 1a: Metrics ===
Config A: recall@1=0.6167, recall@3=0.7667, recall@5=0.8333, MRR@10=0.7046
Config B: recall@1=0.3333, recall@3=0.5167, recall@5=0.7333, MRR@10=0.4570

=== PART 1b: Wilson Intervals ===
Candidate A, k=3: recall=0.7667, lower=0.6456, upper=0.8556
Candidate A, k=5: recall=0.8333, lower=0.7197, upper=0.9069
Candidate B, k=3: recall=0.5167, lower=0.3931, upper=0.6383
Candidate B, k=5: recall=0.7333, lower=0.6099, upper=0.8287

=== PART 1c & 1d: Language Recalls & Costs ===
Cost A, k=3: $18489.41
Cost A, k=5: $25231.96
Cost B, k=3: $11173.19
Cost B, k=5: $13038.26
Recall A k=3 [EN]: 0.9000
Recall A k=5 [EN]: 0.9000
Recall A k=3 [RU]: 0.7000
Recall A k=5 [RU]: 0.8000
Recall A k=3 [KK]: 0.7000
Recall A k=5 [KK]: 0.8000
Recall B k=3 [EN]: 0.6500
Recall B k=5 [EN]: 0.8500
Recall B k=3 [RU]: 0.5000
Recall B k=5 [RU]: 0.7000
Recall B k=3 [KK]: 0.4000
Recall B k=5 [KK]: 0.6500

=== PART 2a: Doc Pipeline ===
Pipeline H Overall Acc: 0.8778 (158/180)
Pipeline L Overall Acc: 0.

`ranks` and `docs` are ready for you to compute with. Remember the **fingerprint** at the top of the data sheet: it goes into the first lines of `ANSWERS.md`.